
Cell 1: Initialization & Environment Setup
Sets up imports, navigates to the repository root directory, and configures output paths for the experiment results. It also creates target folders (stripped_data, pseudocode_classes, and analysis).

In [1]:
import os
import json
import shutil
from datetime import datetime
from pathlib import Path


# --- Robust Path Navigation ---
while Path.cwd().name != 'llamea-cameralens-workspace' and Path.cwd().parent != Path.cwd():
    os.chdir('..')

if Path.cwd().name != 'llamea-cameralens-workspace':
    print(f"⚠️ Warning: Could not find 'llamea-cameralens-workspace'. Current dir: {Path.cwd()}")
else:
    print(f"✅ Successfully navigated to workspace root: {Path.cwd()}")

# --- Configuration ---
INPUT_DIR = Path("blade-framework/results/Lens_v5_50000_False_03_06/llamea_run_lens_v5_50000_F_03_06")
EXPERIMENT_NAME = "lens_v5_50000_False_v3"


current_date = datetime.now().strftime("%d-%m-%Y")
OUTPUT_BASE = Path(f"blade-framework/output-analysis/{EXPERIMENT_NAME}_{current_date}")
SUBFOLDERS = ["stripped_data", "pseudocode_classes", "analysis"]

if OUTPUT_BASE.exists():
    already_exists = True
else:
    already_exists = False
    for folder in SUBFOLDERS:
        (OUTPUT_BASE / folder).mkdir(parents=True, exist_ok=True)

print(f"✅ Setup complete. Output base: {OUTPUT_BASE}")

✅ Successfully navigated to workspace root: /Users/Yotam/Downloads/thesis_temporary_code/lens_v4_conitnuation/llamea-cameralens-workspace
✅ Setup complete. Output base: blade-framework/output-analysis/lens_v5_50000_False_v3_08-06-2026



Cell 2: Parsing & Extracting Code
Parses the experiment's log files (log.jsonl and conversationlog.jsonl) to isolate each optimizer code entry into its own Python file, outputting them to stripped_data/ and saving a metadata registry (variables.json).

In [2]:
def load_jsonl(filepath):
    data = []
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                data.append(json.loads(line))
    return data
variables_path = OUTPUT_BASE / "stripped_data/variables.json"
if variables_path.exists():
    print(f"✅ Variables file already exists at {variables_path}.")
    response = input("Do you want to overwrite it? (y/n): ").strip().lower()
    if response != 'y':
        print("Exiting without overwriting.")
        exit(0)

log_path = INPUT_DIR / "log.jsonl"
conv_log_path = INPUT_DIR / "conversationlog.jsonl"

if log_path.exists():
    logs = load_jsonl(log_path)
else:
    raise FileNotFoundError(f"Log file not found: {log_path}")

if conv_log_path.exists():
    conv_logs = load_jsonl(conv_log_path)
else:
    raise FileNotFoundError(f"Conversation log file not found: {conv_log_path}")

merged_data = {}
for entry in logs:
    entry_id = entry.get("id")
    if entry_id:
        merged_data[entry_id] = {
            "id": entry_id,
            "generation": entry.get("generation", 0),
            "fitness": entry.get("fitness"),
            "parent": entry.get("parent_ids", []),
            "code": entry.get("code", ""),
            "feedback": entry.get("feedback", ""),
            "prompts": []
        }

for conv in conv_logs:
    conv_id = conv.get("id") or conv.get("run_id")
    if conv_id and conv_id in merged_data:
        merged_data[conv_id]["prompts"].append(conv)

print(f"Successfully mapped {len(merged_data)} optimization class entries.")

stripped_dir = OUTPUT_BASE / "stripped_data"
variables_data = {}
isolated_count = 0

for entry_id, entry in merged_data.items():
    code_content = entry.get("code", "")
    if code_content.strip():
        py_filename = f"optimisation_{entry_id}.py"
        file_path = stripped_dir / py_filename
        with open(file_path, "w", encoding="utf-8") as f:
            f.write(code_content)
        isolated_count += 1
        
        variables_data[entry_id] = {
            "id": entry_id,
            "generation": entry["generation"],
            "fitness": entry["fitness"],
            "parent": entry.get("parent", []),
            "feedback": entry["feedback"],
            "code":entry["code"],
            "local_path": str(file_path.absolute())
        }

variables_path = stripped_dir / "variables.json"
with open(variables_path, "w", encoding="utf-8") as f:
    json.dump(variables_data, f, indent=4)

print(f"📂 Isolated and saved {isolated_count} classes and variables.json to: {stripped_dir}")

✅ Variables file already exists at blade-framework/output-analysis/lens_v5_50000_False_v3_08-06-2026/stripped_data/variables.json.
Successfully mapped 12 optimization class entries.
📂 Isolated and saved 12 classes and variables.json to: blade-framework/output-analysis/lens_v5_50000_False_v3_08-06-2026/stripped_data


Defines helper functions to strip inline comments and docstrings from Python files without altering line numbers, and parses indentation levels to isolate nested leaf blocks for bottom-up translation.

In [3]:
import textwrap

def is_empty_or_comment(line):
    s = line.strip()
    return not s or s.startswith('#') or s.startswith('//')

def get_indent(line):
    s = line.expandtabs(4)
    return len(s) - len(s.lstrip())

def strip_comments_and_docstrings(lines):
    """
    Strips inline comments (#) and multiline docstrings (""" """) 
    while replacing them with empty strings to preserve line numbers.
    """
    cleaned_lines = []
    in_docstring = False
    doc_char = ""
    
    for line in lines:
        if in_docstring:
            if doc_char in line:
                in_docstring = False
            # Append empty string to keep line counts identical
            cleaned_lines.append("") 
            continue
        
        stripped_line = line.strip()
        
        # Check for start of multiline docstrings
        if stripped_line.startswith('"""') or stripped_line.startswith("'''"):
            doc_char = stripped_line[:3]
            # Handle single-line docstring (e.g., """ This is a comment """)
            if stripped_line.endswith(doc_char) and len(stripped_line) > 3:
                cleaned_lines.append("")
            else:
                in_docstring = True
                cleaned_lines.append("")
            continue
            
        # Handle inline comments
        if '#' in line:
            # Split at the first hash to remove the comment
            parts = line.split('#')
            clean_line = parts[0].rstrip()
            cleaned_lines.append(clean_line)
        else:
            cleaned_lines.append(line)
            
    return cleaned_lines

def single_pass_translate(lines, canonicalizer=None):
    # 1. Strip comments and docstrings before processing blocks
    lines = strip_comments_and_docstrings(lines)
    
    indents = []
    for i, line in enumerate(lines):
        if not is_empty_or_comment(line):
            indents.append((i, get_indent(line)))
            
    blocks = []
    for idx in range(len(indents) - 1):
        i, ind = indents[idx]
        next_i, next_ind = indents[idx+1]
        
        if next_ind > ind:
            end_idx = idx + 1
            has_nested = False
            while end_idx < len(indents) and indents[end_idx][1] > ind:
                if indents[end_idx][1] > next_ind:
                    has_nested = True
                end_idx += 1
                
            if not has_nested:
                start_line = i
                end_line = indents[end_idx-1][0]
                blocks.append((start_line, end_line))
                
    if not blocks:
        if not any(not is_empty_or_comment(l) for l in lines):
            return lines, False
            
        text = "\n".join(lines)
        pseudocode, _ = translate_chunk(text, canonicalizer=canonicalizer)
        pseudo_wrapped = ["// :::PSEUDOCODE:::"] + [f"// {l}" for l in pseudocode.split('\n')] + ["// :::END_PSEUDOCODE:::"]
        return pseudo_wrapped, True
        
    blocks.sort(key=lambda x: x[0], reverse=True)
    new_lines = list(lines)
    
    for start, end in blocks:
        block_lines = lines[start:end+1]
        block_text = "\n".join(block_lines)
        
        pseudocode, _ = translate_chunk(block_text, canonicalizer=canonicalizer)
        base_indent = get_indent(lines[start])
        prefix = " " * base_indent
        
        pseudo_wrapped = [f"{prefix}// :::PSEUDOCODE:::"]
        for p_line in pseudocode.split('\n'):
            pseudo_wrapped.append(f"{prefix}// {p_line}")
        pseudo_wrapped.append(f"{prefix}// :::END_PSEUDOCODE:::")
        
        new_lines[start:end+1] = pseudo_wrapped
        
    return new_lines, True

In [4]:
import urllib.request
import urllib.error
import hashlib
import time

OLLAMA_URL = "http://localhost:11434/api/chat"
DEFAULT_MODEL = "qwen2.5-coder:14b"
FALLBACK_MODEL = "mistral:latest"

def query_ollama(prompt, system_prompt=None, model=DEFAULT_MODEL, retries=3, delay=2):
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": prompt})
    
    data = {
        "model": model,
        "messages": messages,
        "stream": False
    }
    
    req = urllib.request.Request(
        OLLAMA_URL,
        data=json.dumps(data).encode("utf-8"),
        headers={"Content-Type": "application/json"}
    )
    
    for attempt in range(1, retries + 1):
        try:
            with urllib.request.urlopen(req, timeout=240) as response:
                res_body = response.read().decode("utf-8")
                res_json = json.loads(res_body)
                return res_json["message"]["content"]
        except Exception as e:
            print(f"⚠️ [Ollama Attempt {attempt} Error]: {e}")
            if attempt == retries:
                return query_ollama(prompt, system_prompt, model=FALLBACK_MODEL, retries=retries, delay=delay) if model == DEFAULT_MODEL else ""
            time.sleep(delay * (2 ** (attempt - 1)))

translation_cache = {}
def get_chunk_hash(source):
    return hashlib.sha256(source.encode("utf-8")).hexdigest()

def translate_chunk(chunk_text, canonicalizer=None):
    chunk_hash = get_chunk_hash(chunk_text)
    if chunk_hash in translation_cache:
        return translation_cache[chunk_hash], True
        
    if canonicalizer is not None:
        clean_python = "\n".join(line for line in chunk_text.splitlines() if not line.strip().startswith("//") and not line.strip().startswith("#"))
        canonicalizer.discover_variables(clean_python)
        abstracted_chunk = canonicalizer.canonicalize(chunk_text, discover=False)
    else:
        abstracted_chunk = chunk_text
        
    system_prompt = (
        "You are the PseudocodeGeneratorAgent.\n"
        "Your task is to rewrite a Python block into highly readable, standardized pseudocode.\n\n"
        "IMPORTANT: The Python code you receive has been abstracted. It contains tokens like [VAR_X] and [OP_XXXX]. "
        "You MUST preserve these exact tokens in your output. Do NOT translate, rename, or strip them.\n"
        "The Python code may also contain nested blocks that have ALREADY been translated into pseudocode. "
        "These translated blocks will be wrapped in `// :::PSEUDOCODE:::` and `// :::END_PSEUDOCODE:::` comments. "
        "You MUST seamlessly integrate these existing pseudocode blocks into your final translation of the surrounding Python code.\n\n"
        "RULES:\n"
        "1. Math Notation: Translate optimization math into LaTeX, but keep [VAR_X] intact.\n"
        "2. Token Preservation: Keep all [VAR_0], [VAR_1], [OP_BOUND], [OP_TYPECAST] tokens exactly as written.\n"
        "3. Variable Preservation: Preserve any other exact variable names if they are not tokenized.\n"
        "4. Structural Mapping: Replace conditional constructs with uppercase statements (IF/THEN/ELSE, FOR, WHILE).\n"
        "Return ONLY the pseudocode block. Do not add conversational text."
    )
    
    prompt = f"Translate the following abstracted code block into pseudocode:\n\n```python\n{abstracted_chunk}\n```"
    translated_pseudocode = query_ollama(prompt, system_prompt)
    translation_cache[chunk_hash] = translated_pseudocode
    return translated_pseudocode, False

In [5]:
import sys
from pathlib import Path

# Add src to sys.path to import Canonicalizer
src_path = Path.cwd() / "blade-framework" / "output-analysis" / "src"
if str(src_path) not in sys.path:
    sys.path.append(str(src_path))

from canonicalizer import Canonicalizer

variables_path = stripped_dir / "variables.json"
pseudocode_dir = OUTPUT_BASE / "pseudocode_classes"

# Open and load the JSON file
with open(variables_path, "r", encoding="utf-8") as f:
    loaded_data = json.load(f)

# Iterate through the loaded dictionary
for entry_id, entry in loaded_data.items():
    code_content = entry.get("code", "")
    if not code_content.strip(): continue
    
    # Initialize Canonicalizer for this class and pre-discover variables
    canonicalizer = Canonicalizer(OUTPUT_BASE, entry_id)
    canonicalizer.discover_variables(code_content)
    
    class_dir = pseudocode_dir / f"class_{entry_id}"
    class_dir.mkdir(parents=True, exist_ok=True)
    
    current_lines = code_content.split('\n')
    iteration = 1
    
    print(f"🚀 Starting Recursive Translation for Optimizer {entry_id[:8]}...")
    while True:
        # Pass the canonicalizer down
        new_lines, changed = single_pass_translate(current_lines, canonicalizer=canonicalizer)
        
        if not changed:
            break
            
        iter_path = class_dir / f"iteration_{iteration}.md"
        with open(iter_path, "w", encoding="utf-8") as f:
            f.write("\n".join(new_lines))
            
        print(f"   ✅ Saved iteration {iteration} to {iter_path.name}")
        current_lines = new_lines
        iteration += 1
        
print(f"✨ All recursive translations complete. Data saved to {pseudocode_dir}")

🚀 Starting Recursive Translation for Optimizer ee684047...
   ✅ Saved iteration 1 to iteration_1.md
   ✅ Saved iteration 2 to iteration_2.md
   ✅ Saved iteration 3 to iteration_3.md
   ✅ Saved iteration 4 to iteration_4.md
   ✅ Saved iteration 5 to iteration_5.md
🚀 Starting Recursive Translation for Optimizer 21a3d62d...
   ✅ Saved iteration 1 to iteration_1.md
   ✅ Saved iteration 2 to iteration_2.md
   ✅ Saved iteration 3 to iteration_3.md
   ✅ Saved iteration 4 to iteration_4.md
   ✅ Saved iteration 5 to iteration_5.md
   ✅ Saved iteration 6 to iteration_6.md
   ✅ Saved iteration 7 to iteration_7.md
🚀 Starting Recursive Translation for Optimizer c9d0657c...
   ✅ Saved iteration 1 to iteration_1.md
   ✅ Saved iteration 2 to iteration_2.md
   ✅ Saved iteration 3 to iteration_3.md
   ✅ Saved iteration 4 to iteration_4.md
   ✅ Saved iteration 5 to iteration_5.md
   ✅ Saved iteration 6 to iteration_6.md
   ✅ Saved iteration 7 to iteration_7.md
🚀 Starting Recursive Translation for Optimiz

In [6]:
def extract_clean_markdown(lines):
    clean = []
    for line in lines:
        s = line.strip()
        
        # Skip empty lines
        if not s:
            continue
            
        if s in ["// :::PSEUDOCODE:::", "// :::END_PSEUDOCODE:::"]:
            continue
            
        if line.lstrip().startswith("// "):
            indent = len(line) - len(line.lstrip())
            clean.append(" " * indent + line.lstrip()[3:])
        else:
            clean.append(line)
            
    return clean

variables_path = OUTPUT_BASE / "stripped_data/variables.json"
pseudocode_dir = OUTPUT_BASE / "pseudocode_classes"

# Open and load the JSON file
with open(variables_path, "r", encoding="utf-8") as f:
    loaded_data = json.load(f)

# Iterate through the loaded dictionary
for entry_id, entry in loaded_data.items():

    class_dir = pseudocode_dir / f"class_{entry_id}"
    if not class_dir.exists(): continue
    
    iter_files = list(class_dir.glob("iteration_*.md"))
    if not iter_files: continue
    
    latest_file = max(iter_files, key=lambda f: int(f.stem.split('_')[1]))
    
    with open(latest_file, "r", encoding="utf-8") as f:
        final_lines = f.read().split('\n')
        
    clean_lines = extract_clean_markdown(final_lines)
    
    final_md_path = class_dir / "final_pseudocode.md"
    with open(final_md_path, "w", encoding="utf-8") as f:
        f.write("\n".join(clean_lines))
        
print("✨ Clean final pseudocode extracted successfully.")

✨ Clean final pseudocode extracted successfully.


In [7]:
import sys
from pathlib import Path

# Add src to sys.path to import Canonicalizer
src_path = Path.cwd() / "blade-framework" / "output-analysis" / "src"
if str(src_path) not in sys.path:
    sys.path.append(str(src_path))

from canonicalizer import Canonicalizer

pseudocode_dir = OUTPUT_BASE / "pseudocode_classes"
def extract_clean_markdown(lines):
    clean = []
    for line in lines:
        s = line.strip()
        # Skip the pseudocode boundary stickers
        if s in ["// :::PSEUDOCODE:::", "// :::END_PSEUDOCODE:::"]:
            continue
        # Strip the "// " prefix from LLM outputs while preserving indentation
        if line.lstrip().startswith("// "):
            indent = len(line) - len(line.lstrip())
            clean.append(" " * indent + line.lstrip()[3:])
        else:
            clean.append(line)
    return clean

for entry_id, entry in loaded_data.items():
    class_dir = pseudocode_dir / f"class_{entry_id}"
    if not class_dir.exists(): continue
    
    # Find all iteration files for this class
    iter_files = list(class_dir.glob("iteration_*.md"))
    iter_files = [f for f in iter_files if f.name != "Iteration_Final.md"]
    if not iter_files: continue
    
    # Identify the final (highest number) iteration file
    latest_file = max(iter_files, key=lambda f: int(f.stem.split('_')[1]))
    
    # Read the final raw text
    with open(latest_file, "r", encoding="utf-8") as f:
        final_lines = f.read().split('\n')
        
    # Clean the stickers
    clean_lines = extract_clean_markdown(final_lines)
    clean_text = "\n".join(clean_lines)
    
    # Run canonicalization at the very end
    canonicalizer = Canonicalizer(OUTPUT_BASE, entry_id)
    # Discover variables from original Python code
    canonicalizer.discover_variables(entry["code"])
    # Canonicalize the final pseudocode
    canonicalized_text = canonicalizer.canonicalize(clean_text, discover=False)
    
    # Save as Iteration_Final.md
    final_md_path = class_dir / "Iteration_Final.md"
    with open(final_md_path, "w", encoding="utf-8") as f:
        f.write(canonicalized_text)
        
print("✨ Clean Iteration_Final.md extracted and canonicalized successfully for all classes.")

✨ Clean Iteration_Final.md extracted and canonicalized successfully for all classes.


In [8]:
import sys
from pathlib import Path

# Add the src directory to the system path
sys.path.append(str(Path.cwd() / "blade-framework" / "output-analysis" / "src"))

from translation_engine import TranslationEngine
# 1. Create fixture files to test edge cases
os.makedirs('tests/fixtures', exist_ok=True)

class1_content = """\
def optimize(eval_x):
    eval_x[18:24] = CONVERT_TO_INT(CLIP(ROUND(eval_x[18:24]), 0, 5))
    return eval_x
"""
with open('tests/fixtures/class_1.py', 'w') as f:
    f.write(class1_content)

class2_content = """\
def optimize(eval_x):
    # Note the different spacing and CAST instead of CONVERT_TO_INT
    eval_x[18:24] = CAST(ROUND(eval_x[18:24]), INTEGER)
    return eval_x
"""
with open('tests/fixtures/class_2.py', 'w') as f:
    f.write(class2_content)

class3_content = """\
def optimize(eval_x):
    eval_x[18:24] = CONVERT_TO_INT(CLIP(ROUND(eval_x[18:24]), 0, 5))
    eval_x[18:24] = CONVERT_TO_INT(CLIP(ROUND(eval_x[18:24]), 0, 5))
    return eval_x
"""
with open('tests/fixtures/class_3.py', 'w') as f:
    f.write(class3_content)

print("Fixtures created successfully!")


ModuleNotFoundError: No module named 'requests'